# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahasuhail27-cyber/AI-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

### Baseline Rule

The baseline rule ranks content based on historical search performance.

The rule prioritizes content that has:
- High search impressions (good opportunity)
- Low search clicks (underperforming)
- Poor average search position

These signals suggest content that could benefit from optimization.

### Reason Codes

**LOW_CTR**
- High impressions but relatively few clicks.

**LOW_POSITION**
- Poor average Google Search position.

**HIGH_OPPORTUNITY**
- High visibility with potential for improvement.

The baseline score is intended as a decision-support tool rather than a prediction.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
import pandas as pd
import os

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN").strip()

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret(
TYPE HUGGINGFACE,
TOKEN '{HF_TOKEN}'
)
""")

print("Connected successfully")

Connected successfully


## 2. Build the ranked queue (writes the CSV)

A simple baseline score is created using historical search metrics.

Higher impressions increase opportunity, while fewer clicks and poorer search position increase priority.

Each content item receives:
- Baseline Score
- Reason Code
- Action Label

The ranked queue is written to:

`work/outputs/baseline_action_score.csv`

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

os.makedirs("work/outputs", exist_ok=True)

baseline = con.sql("""
SELECT
content_hash_id,
client_hash_id,
gsc_impressions,
gsc_clicks,
gsc_avg_position,

(gsc_impressions*0.4)
+
((100-gsc_clicks)*0.3)
+
(gsc_avg_position*0.3)

AS baseline_score,

CASE
WHEN gsc_clicks < 10 THEN 'LOW_CTR'
WHEN gsc_avg_position > 20 THEN 'LOW_POSITION'
ELSE 'HIGH_OPPORTUNITY'
END AS reason_code,

CASE
WHEN gsc_clicks < 10 THEN 'Improve Title / Meta Description'
WHEN gsc_avg_position > 20 THEN 'Improve SEO Content'
ELSE 'Monitor'
END AS action

FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
)

WHERE gsc_data_available IS TRUE

ORDER BY baseline_score DESC

LIMIT 100
""").df()

baseline.to_csv(
"work/outputs/baseline_action_score.csv",
index=False
)

baseline.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,client_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,baseline_score,reason_code,action
0,content_44f34c0a90047651,client_23a62021009f63c4,40084,1,0.083350,16063.325005,LOW_CTR,Improve Title / Meta Description
1,content_eadb33b5df496f4a,client_e547b89c05043229,39305,252,2.197507,15677.059252,HIGH_OPPORTUNITY,Monitor
2,content_34a70fea29d15f24,client_62f4a7e64f5e0096,39003,2,2.764916,15631.429475,LOW_CTR,Improve Title / Meta Description
3,content_eadb33b5df496f4a,client_e547b89c05043229,38436,271,2.195988,15323.758796,HIGH_OPPORTUNITY,Monitor
4,content_945d6ff91386c817,client_62f4a7e64f5e0096,37368,0,8.613948,14979.784184,LOW_CTR,Improve Title / Meta Description


## 3. Top-20 review

The highest-ranked content items were manually reviewed.

For each item:

- Action: based on the rule output.
- Reason Code: explains why it was prioritized.
- Confidence: Medium, because the rule uses only historical search signals.
- What would make it wrong:
  - Seasonal traffic
  - Recently updated pages
  - Temporary ranking fluctuations
  - Missing analytics data

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = baseline.head(20)

top20

,content_hash_id,client_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,baseline_score,reason_code,action
0,content_44f34c0a90047651,client_23a62021009f63c4,40084,1,0.083350,16063.325005,LOW_CTR,Improve Title / Meta Description
1,content_eadb33b5df496f4a,client_e547b89c05043229,39305,252,2.197507,15677.059252,HIGH_OPPORTUNITY,Monitor
2,content_34a70fea29d15f24,client_62f4a7e64f5e0096,39003,2,2.764916,15631.429475,LOW_CTR,Improve Title / Meta Description
3,content_eadb33b5df496f4a,client_e547b89c05043229,38436,271,2.195988,15323.758796,HIGH_OPPORTUNITY,Monitor
4,content_945d6ff91386c817,client_62f4a7e64f5e0096,37368,0,8.613948,14979.784184,LOW_CTR,Improve Title / Meta Description
5,content_eadb33b5df496f4a,client_e547b89c05043229,35404,225,2.188397,14124.756519,HIGH_OPPORTUNITY,Monitor
6,content_eadb33b5df496f4a,client_e547b89c05043229,34817,223,2.181348,13890.554404,HIGH_OPPORTUNITY,Monitor
7,content_eadb33b5df496f4a,client_e547b89c05043229,34606,235,2.242501,13802.572750,HIGH_OPPORTUNITY,Monitor
8,content_eadb33b5df496f4a,client_e547b89c05043229,33571,215,2.309046,13394.592714,HIGH_OPPORTUNITY,Monitor
9,content_fec55986a1868d62,client_73cda7b4e4f265ea,33383,0,0.181500,13383.254450,LOW_CTR,Improve Title / Meta Description


## 4. Weak picks + leakage check

Some ranked items may not actually require optimization.

Possible reasons include:

- Seasonal search behaviour.
- Missing Search Console or Analytics data.
- Recent content updates not reflected in historical metrics.

Leakage Check:

No future information was used.

No label-derived fields were included.

Only historical Search Console metrics available at the decision time were used.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
baseline.tail(10)

,content_hash_id,client_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,baseline_score,reason_code,action
90,content_eadb33b5df496f4a,client_e547b89c05043229,12111,140,2.542565,4833.162769,HIGH_OPPORTUNITY,Monitor
91,content_b99ea6861864dea5,client_62f4a7e64f5e0096,11965,16,4.924530,4812.677359,HIGH_OPPORTUNITY,Monitor
92,content_bf078007df823490,client_23a62021009f63c4,11952,0,1.063922,4811.119177,LOW_CTR,Improve Title / Meta Description
93,content_eadb33b5df496f4a,client_e547b89c05043229,12027,107,2.452648,4809.435794,HIGH_OPPORTUNITY,Monitor
94,content_4ffe18112a5642e3,client_e547b89c05043229,11946,25,2.471873,4801.641562,HIGH_OPPORTUNITY,Monitor
95,content_ec2e0346994fb5a5,client_e547b89c05043229,11966,58,3.029667,4799.908900,HIGH_OPPORTUNITY,Monitor
96,content_4ffe18112a5642e3,client_e547b89c05043229,11913,21,2.488542,4789.646563,HIGH_OPPORTUNITY,Monitor
97,content_4ffe18112a5642e3,client_e547b89c05043229,11751,20,2.381670,4725.114501,HIGH_OPPORTUNITY,Monitor
98,content_04fb6296acff6360,client_e547b89c05043229,11679,1,1.779776,4701.833933,LOW_CTR,Improve Title / Meta Description
99,content_4ffe18112a5642e3,client_e547b89c05043229,11615,26,2.448041,4668.934412,HIGH_OPPORTUNITY,Monitor


## Self-check

## Self-check

- ✅ Every section is completed.
- ✅ Notebook runs without errors.
- ✅ No private client information is exposed.
- ✅ Only historical features were used.
- ✅ No future-window or label-derived information was included.
- ✅ CSV successfully written to `work/outputs/baseline_action_score.csv`.